# PISA 2022 & HSLS:09 Interactive Visualizations (Redesigned v2)

Chart types restricted to: **Line, Scatter, Horizontal Bar, Vertical Bar, Heat Map**

- **Viz 1-3:** PISA-only charts
- **Viz 4-6:** HSLS-only charts
- **Viz 7-9:** Combined charts (highlight for 7-8, filter for 9)

In [ ]:
import pandas as pd
import altair as alt
from pathlib import Path
import json
import numpy as np

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../assets/json")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pisa_df = pd.read_csv(DATA_DIR / "pisa_subset.csv", low_memory=False)
hsls_df = pd.read_csv(DATA_DIR / "hsls_subset.csv", low_memory=False)

print(f"PISA: {len(pisa_df)} rows, {len(pisa_df.columns)} columns")
print(f"HSLS: {len(hsls_df)} rows, {len(hsls_df.columns)} columns")

In [ ]:
DARK_CONFIG = {
    "background": "#030712",
    "view": {"stroke": "transparent"},
    "axis": {
        "labelColor": "#E0E0E0",
        "titleColor": "#FFFFFF",
        "gridColor": "#333333",
        "domainColor": "#444444",
        "tickColor": "#444444"
    },
    "legend": {"labelColor": "#E0E0E0", "titleColor": "#FFFFFF"},
    "title": {"color": "#FFFFFF"}
}

def save_chart(chart, filename):
    spec = json.loads(chart.to_json())
    spec["config"] = DARK_CONFIG
    with open(OUTPUT_DIR / filename, "w") as f:
        json.dump(spec, f, indent=2)
    print(f"Saved: {filename}")

---
## PISA-ONLY VISUALIZATIONS (Viz 1-3)
---

### Viz 1: Gender Confidence Gap -> Achievement Gap by Percentile

**Driver (Horizontal Bar):** Countries ranked by efficacy gap (Male - Female)  
**Driven (Line Chart):** Achievement gap at percentiles 10, 25, 50, 75, 90 for selected country

In [ ]:
pisa_gender = pisa_df[(pisa_df["MATHEFF"].notna()) & 
                       (pisa_df["PV1MATH"].notna()) & 
                       (pisa_df["ST004D01T"].isin([1, 2]))].copy()
pisa_gender["Gender"] = pisa_gender["ST004D01T"].map({1: "Female", 2: "Male"})

efficacy_by_country = pisa_gender.groupby(["CNT", "Gender"])["MATHEFF"].mean().unstack()
efficacy_by_country["Efficacy_Gap"] = efficacy_by_country["Male"] - efficacy_by_country["Female"]
top_countries = efficacy_by_country.nlargest(15, "Efficacy_Gap").index.tolist()

driver_data = efficacy_by_country.loc[top_countries].reset_index()
driver_data = driver_data[["CNT", "Efficacy_Gap"]].rename(columns={"CNT": "Country", "Efficacy_Gap": "Gap"})

percentiles = [10, 25, 50, 75, 90]
line_data = []
for country in top_countries:
    country_df = pisa_gender[pisa_gender["CNT"] == country]
    for p in percentiles:
        female_q = country_df[country_df["Gender"] == "Female"]["PV1MATH"].quantile(p/100)
        male_q = country_df[country_df["Gender"] == "Male"]["PV1MATH"].quantile(p/100)
        line_data.append({"Country": country, "Percentile": p, "Achievement_Gap": male_q - female_q})
line_df = pd.DataFrame(line_data)

country_select = alt.selection_point(fields=["Country"], name="country_select")

left_chart = alt.Chart(driver_data).mark_bar(cornerRadiusTopRight=4, cornerRadiusBottomRight=4, cursor="pointer").encode(
    x=alt.X("Gap:Q", title="Efficacy Gap (Male - Female)", scale=alt.Scale(domain=[0, 0.6])),
    y=alt.Y("Country:N", sort=alt.EncodingSortField(field="Gap", order="descending"), title=None),
    color=alt.value("#4c8dff"),
    opacity=alt.condition(country_select, alt.value(1), alt.value(0.3)),
    tooltip=["Country:N", alt.Tooltip("Gap:Q", format=".3f", title="Efficacy Gap")]
).add_params(country_select).properties(
    name="view_1",
    title={"text": "Click Country: Confidence Gap", "color": "#FFFFFF", "fontSize": 14},
    width=450, height=350
)

right_chart = alt.Chart(line_df).mark_line(point={"filled": True, "size": 80}, strokeWidth=3).encode(
    x=alt.X("Percentile:O", title="Score Percentile", axis=alt.Axis(labelAngle=0)),
    y=alt.Y("Achievement_Gap:Q", title="Achievement Gap (Male - Female)", scale=alt.Scale(domain=[-30, 50])),
    color=alt.value("#a855f7"),
    tooltip=["Country:N", "Percentile:O", alt.Tooltip("Achievement_Gap:Q", format=".1f")]
).transform_filter(country_select).properties(
    title={"text": "Achievement Gap by Percentile", "color": "#FFFFFF", "fontSize": 14},
    width=450, height=350
)

viz1 = alt.hconcat(left_chart, right_chart).resolve_scale(color="independent")
save_chart(viz1, "pisa_gender_efficacy_dumbbell.json")
viz1

### Viz 2: Anxiety Level -> Achievement Distribution

**Driver (Heat Map):** Anxiety bins x Gender -> mean Math score  
**Driven (Scatter + LOESS):** ANXMAT vs PV1MATH colored by Gender

In [ ]:
pisa_anxiety = pisa_df[(pisa_df["ANXMAT"].notna()) & 
                        (pisa_df["PV1MATH"].notna()) &
                        (pisa_df["ST004D01T"].isin([1, 2]))].copy()
pisa_anxiety["Gender"] = pisa_anxiety["ST004D01T"].map({1: "Female", 2: "Male"})
pisa_anxiety["Anxiety_Bin"] = pd.qcut(pisa_anxiety["ANXMAT"], 5, labels=["Very Low", "Low", "Medium", "High", "Very High"])

heatmap_data = pisa_anxiety.groupby(["Anxiety_Bin", "Gender"]).agg({"PV1MATH": "mean"}).reset_index()
heatmap_data.columns = ["Anxiety_Bin", "Gender", "Mean_Math"]
heatmap_data["Anxiety_Bin"] = heatmap_data["Anxiety_Bin"].astype(str)

scatter_data = pisa_anxiety[["ANXMAT", "PV1MATH", "Gender", "Anxiety_Bin"]].sample(n=4000, random_state=42)
scatter_data.columns = ["Anxiety", "Math_Score", "Gender", "Anxiety_Bin"]
scatter_data["Anxiety_Bin"] = scatter_data["Anxiety_Bin"].astype(str)

anxiety_order = ["Very Low", "Low", "Medium", "High", "Very High"]
anxiety_select = alt.selection_point(fields=["Anxiety_Bin"], name="anxiety_select")

left_chart = alt.Chart(heatmap_data).mark_rect(cursor="pointer").encode(
    x=alt.X("Gender:N", title=None),
    y=alt.Y("Anxiety_Bin:O", title="Math Anxiety Level", sort=anxiety_order),
    color=alt.Color("Mean_Math:Q", scale=alt.Scale(scheme="viridis"), title="Mean Math Score"),
    opacity=alt.condition(anxiety_select, alt.value(1), alt.value(0.3)),
    tooltip=["Anxiety_Bin:N", "Gender:N", alt.Tooltip("Mean_Math:Q", format=".1f")]
).add_params(anxiety_select).properties(
    name="view_1",
    title={"text": "Click Anxiety Level", "color": "#FFFFFF", "fontSize": 14},
    width=200, height=350
)

scatter_layer = alt.Chart(scatter_data).mark_circle(opacity=0.4, size=30).encode(
    x=alt.X("Anxiety:Q", title="Math Anxiety (continuous)", scale=alt.Scale(domain=[-3, 3])),
    y=alt.Y("Math_Score:Q", title="Math Score", scale=alt.Scale(domain=[200, 800])),
    color=alt.Color("Gender:N", scale=alt.Scale(domain=["Female", "Male"], range=["#E91E63", "#1976D2"]),
                    legend=alt.Legend(title="Gender", orient="top")),
    tooltip=["Gender:N", alt.Tooltip("Anxiety:Q", format=".2f"), alt.Tooltip("Math_Score:Q", format=".0f")]
).transform_filter(anxiety_select)

loess_layer = alt.Chart(scatter_data).mark_line(strokeWidth=3).encode(
    x=alt.X("Anxiety:Q"),
    y=alt.Y("Math_Score:Q"),
    color=alt.Color("Gender:N", scale=alt.Scale(domain=["Female", "Male"], range=["#E91E63", "#1976D2"]))
).transform_filter(anxiety_select).transform_loess(on="Anxiety", loess="Math_Score", groupby=["Gender"])

right_chart = (scatter_layer + loess_layer).properties(
    title={"text": "Anxiety vs Achievement", "color": "#FFFFFF", "fontSize": 14},
    width=450, height=350
)

viz2 = alt.hconcat(left_chart, right_chart).resolve_scale(color="independent")
save_chart(viz2, "pisa_anxiety_performance_heatmap.json")
viz2

### Viz 3: Immigration Status -> Belonging & Achievement

**Driver (Vertical Bar):** Mean math score by immigration status  
**Driven (Scatter):** BELONG vs Math colored by Gender

In [ ]:
pisa_immig = pisa_df[(pisa_df["IMMIG"].isin([1, 2, 3])) & 
                      (pisa_df["PV1MATH"].notna()) &
                      (pisa_df["BELONG"].notna()) &
                      (pisa_df["ST004D01T"].isin([1, 2]))].copy()
pisa_immig["Immigration_Status"] = pisa_immig["IMMIG"].map({1: "Native", 2: "Second-gen", 3: "First-gen"})
pisa_immig["Gender"] = pisa_immig["ST004D01T"].map({1: "Female", 2: "Male"})

immig_agg = pisa_immig.groupby("Immigration_Status").agg({"PV1MATH": "mean", "IMMIG": "count"}).reset_index()
immig_agg.columns = ["Immigration_Status", "Avg_Math", "Count"]

scatter_data = pisa_immig[["Immigration_Status", "Gender", "BELONG", "PV1MATH"]].sample(n=5000, random_state=42)
scatter_data.columns = ["Immigration_Status", "Gender", "Belonging", "Math_Score"]

immig_order = ["Native", "Second-gen", "First-gen"]
immig_colors = ["#0072B2", "#56B4E9", "#D55E00"]
immig_select = alt.selection_point(fields=["Immigration_Status"], name="immig_select")

left_chart = alt.Chart(immig_agg).mark_bar(cornerRadiusTopLeft=4, cornerRadiusTopRight=4, cursor="pointer").encode(
    x=alt.X("Immigration_Status:N", title=None, sort=immig_order, axis=alt.Axis(labelAngle=-45)),
    y=alt.Y("Avg_Math:Q", title="Mean Math Score", scale=alt.Scale(domain=[400, 500])),
    color=alt.Color("Immigration_Status:N", scale=alt.Scale(domain=immig_order, range=immig_colors),
                    legend=alt.Legend(title="Immigration Status", orient="top")),
    opacity=alt.condition(immig_select, alt.value(1), alt.value(0.3)),
    tooltip=["Immigration_Status:N", alt.Tooltip("Avg_Math:Q", format=".1f"), alt.Tooltip("Count:Q", format=",d")]
).add_params(immig_select).properties(
    name="view_1",
    title={"text": "Click Immigration Status", "color": "#FFFFFF", "fontSize": 14},
    width=250, height=350
)

right_chart = alt.Chart(scatter_data).mark_circle(size=30, opacity=0.5).encode(
    x=alt.X("Belonging:Q", title="Sense of Belonging", scale=alt.Scale(domain=[-3, 3])),
    y=alt.Y("Math_Score:Q", title="Math Achievement", scale=alt.Scale(domain=[200, 800])),
    color=alt.Color("Gender:N", scale=alt.Scale(domain=["Female", "Male"], range=["#E91E63", "#1976D2"]),
                    legend=alt.Legend(title="Gender", orient="top")),
    tooltip=["Immigration_Status:N", "Gender:N", alt.Tooltip("Belonging:Q", format=".2f"), alt.Tooltip("Math_Score:Q", format=".0f")]
).transform_filter(immig_select).properties(
    title={"text": "Belonging vs Achievement", "color": "#FFFFFF", "fontSize": 14},
    width=450, height=350
)

viz3 = alt.hconcat(left_chart, right_chart).resolve_scale(color="independent")
save_chart(viz3, "combined_immigration.json")
viz3

---
## HSLS-ONLY VISUALIZATIONS (Viz 4-6)
---

### Viz 4: Parent Education -> STEM Expectations by Gender

**Driver (Line Chart - 2 lines):** STEM rate by parent education, colored by Gender  
**Driven (Vertical Bar):** Gender breakdown for selected education level

In [ ]:
hsls_stem = hsls_df[(hsls_df["X1PAR1EDU"].notna()) & 
                     (hsls_df["X1SEX"].notna()) & 
                     (hsls_df["X1STU30OCC_STEM1"].isin([0, 1]))].copy()
hsls_stem = hsls_stem[(hsls_stem["X1PAR1EDU"] > 0) & (hsls_stem["X1SEX"].isin([1, 2]))]

edu_map = {1: "Less than HS", 2: "HS Diploma", 3: "Some College", 4: "Associate's", 5: "Bachelor's", 6: "Master's", 7: "Ph.D./Prof"}
hsls_stem["Parent_Education"] = hsls_stem["X1PAR1EDU"].map(edu_map)
hsls_stem["Gender"] = hsls_stem["X1SEX"].map({1: "Male", 2: "Female"})

line_data = hsls_stem.groupby(["Parent_Education", "Gender"]).agg(
    STEM_Rate=("X1STU30OCC_STEM1", "mean"),
    Count=("X1SEX", "count")
).reset_index()
line_data["STEM_Rate"] = line_data["STEM_Rate"] * 100

edu_order = ["Less than HS", "HS Diploma", "Some College", "Associate's", "Bachelor's", "Master's", "Ph.D./Prof"]
edu_select = alt.selection_point(fields=["Parent_Education"], name="edu_select")

left_chart = alt.Chart(line_data).mark_line(point={"filled": True, "size": 80}, strokeWidth=3, cursor="pointer").encode(
    x=alt.X("Parent_Education:O", title="Parent Education", sort=edu_order, axis=alt.Axis(labelAngle=-45)),
    y=alt.Y("STEM_Rate:Q", title="STEM Expectation Rate (%)", scale=alt.Scale(domain=[0, 50])),
    color=alt.Color("Gender:N", scale=alt.Scale(domain=["Female", "Male"], range=["#E91E63", "#1976D2"]),
                    legend=alt.Legend(title="Gender", orient="top")),
    opacity=alt.condition(edu_select, alt.value(1), alt.value(0.3)),
    tooltip=["Parent_Education:N", "Gender:N", alt.Tooltip("STEM_Rate:Q", format=".1f"), alt.Tooltip("Count:Q", format=",d")]
).add_params(edu_select).properties(
    name="view_1",
    title={"text": "Click Education Level", "color": "#FFFFFF", "fontSize": 14},
    width=450, height=350
)

right_chart = alt.Chart(line_data).mark_bar(cornerRadiusTopLeft=4, cornerRadiusTopRight=4).encode(
    x=alt.X("Gender:N", title=None),
    y=alt.Y("STEM_Rate:Q", title="STEM Expectation Rate (%)", scale=alt.Scale(domain=[0, 50])),
    color=alt.Color("Gender:N", scale=alt.Scale(domain=["Female", "Male"], range=["#E91E63", "#1976D2"]),
                    legend=alt.Legend(title="Gender", orient="top")),
    tooltip=["Parent_Education:N", "Gender:N", alt.Tooltip("STEM_Rate:Q", format=".1f"), alt.Tooltip("Count:Q", format=",d")]
).transform_filter(edu_select).properties(
    title={"text": "Gender Gap for Selected Level", "color": "#FFFFFF", "fontSize": 14},
    width=250, height=350
)

viz4 = alt.hconcat(left_chart, right_chart).resolve_scale(color="independent")
save_chart(viz4, "combined_gender_stem.json")
viz4

### Viz 5: Race/Ethnicity -> Math Identity vs Achievement

**Driver (Horizontal Bar):** Mean math identity by race  
**Driven (Scatter + regression):** Identity vs Test Score with linear trend

In [ ]:
hsls_identity = hsls_df[(hsls_df["X1MTHID"].notna()) & 
                         (hsls_df["X1TXMTSCOR"].notna()) & 
                         (hsls_df["X1RACE"].notna()) &
                         (hsls_df["X1SEX"].isin([1, 2]))].copy()
hsls_identity = hsls_identity[(hsls_identity["X1RACE"] > 0) & 
                               (hsls_identity["X1MTHID"] >= -3) & 
                               (hsls_identity["X1MTHID"] <= 3) &
                               (hsls_identity["X1TXMTSCOR"] >= 0)]

race_map = {
    1: "Am. Indian/Alaska Native", 2: "Asian", 3: "Black/African American",
    4: "Hispanic", 5: "More than one race", 6: "Native Hawaiian/Pac. Isl.", 7: "White"
}
hsls_identity["Race"] = hsls_identity["X1RACE"].map(race_map)
hsls_identity["Gender"] = hsls_identity["X1SEX"].map({1: "Male", 2: "Female"})

race_agg = hsls_identity.groupby("Race").agg(
    Math_Identity=("X1MTHID", "mean"),
    Count=("X1TXMTSCOR", "count")
).reset_index()

scatter_data = hsls_identity[["Race", "Gender", "X1MTHID", "X1TXMTSCOR"]].sample(n=5000, random_state=42)
scatter_data.columns = ["Race", "Gender", "Math_Identity", "Math_Score"]

race_order = list(race_map.values())
race_select = alt.selection_point(fields=["Race"], name="race_select")

left_chart = alt.Chart(race_agg).mark_bar(cornerRadiusTopRight=4, cornerRadiusBottomRight=4, cursor="pointer").encode(
    x=alt.X("Math_Identity:Q", title="Average Math Identity", scale=alt.Scale(domain=[-0.5, 0.5])),
    y=alt.Y("Race:N", title=None, sort=race_order),
    color=alt.value("#4c8dff"),
    opacity=alt.condition(race_select, alt.value(1), alt.value(0.3)),
    tooltip=["Race:N", alt.Tooltip("Math_Identity:Q", format=".2f"), alt.Tooltip("Count:Q", format=",d", title="n")]
).add_params(race_select).properties(
    name="view_1",
    title={"text": "Click Race/Ethnicity", "color": "#FFFFFF", "fontSize": 14},
    width=450, height=350
)

scatter_layer = alt.Chart(scatter_data).mark_circle(size=30, opacity=0.4).encode(
    x=alt.X("Math_Identity:Q", title="Math Identity Score", scale=alt.Scale(domain=[-3, 3])),
    y=alt.Y("Math_Score:Q", title="Math Test Score", scale=alt.Scale(domain=[20, 80])),
    color=alt.Color("Gender:N", scale=alt.Scale(domain=["Female", "Male"], range=["#E91E63", "#1976D2"]),
                    legend=alt.Legend(title="Gender", orient="top")),
    tooltip=["Race:N", "Gender:N", alt.Tooltip("Math_Identity:Q", format=".2f"), alt.Tooltip("Math_Score:Q", format=".1f")]
).transform_filter(race_select)

regression_layer = alt.Chart(scatter_data).mark_line(strokeWidth=3, strokeDash=[5,5]).encode(
    x=alt.X("Math_Identity:Q"),
    y=alt.Y("Math_Score:Q"),
    color=alt.Color("Gender:N", scale=alt.Scale(domain=["Female", "Male"], range=["#E91E63", "#1976D2"]))
).transform_filter(race_select).transform_regression(on="Math_Identity", regression="Math_Score", groupby=["Gender"])

right_chart = (scatter_layer + regression_layer).properties(
    title={"text": "Identity vs Achievement (descriptive)", "color": "#FFFFFF", "fontSize": 14},
    width=450, height=350
)

viz5 = alt.hconcat(left_chart, right_chart).resolve_scale(color="independent")
save_chart(viz5, "hsls_math_identity_race.json")
viz5

### Viz 6: SES Quintile -> STEM Major Outcomes

**Driver (Vertical Bar):** Mean 12th grade GPA by SES quintile  
**Driven (Heat Map):** SES quintile x STEM Major outcome proportions

In [ ]:
hsls_ses = hsls_df[(hsls_df["X1SESQ5"].notna()) & 
                    (hsls_df["X3TGPA12TH"].notna()) &
                    (hsls_df["X4RFDGMJSTEM"].isin([0, 1]))].copy()
hsls_ses = hsls_ses[(hsls_ses["X1SESQ5"] > 0) & (hsls_ses["X3TGPA12TH"] > 0)]

ses_labels = {1: "Lowest 20%", 2: "Second 20%", 3: "Middle 20%", 4: "Fourth 20%", 5: "Highest 20%"}
hsls_ses["SES_Quintile"] = hsls_ses["X1SESQ5"].map(ses_labels)
hsls_ses["STEM_Major"] = hsls_ses["X4RFDGMJSTEM"].map({1: "STEM", 0: "Non-STEM"})

gpa_data = hsls_ses.groupby("SES_Quintile").agg(GPA=("X3TGPA12TH", "mean"), Count=("X1SESQ5", "count")).reset_index()

stem_counts = hsls_ses.groupby(["SES_Quintile", "STEM_Major"]).size().unstack(fill_value=0)
stem_props = stem_counts.div(stem_counts.sum(axis=1), axis=0).reset_index().melt(
    id_vars=["SES_Quintile"], var_name="STEM_Major", value_name="Proportion"
)
stem_props["Proportion"] = stem_props["Proportion"] * 100

ses_order = ["Lowest 20%", "Second 20%", "Middle 20%", "Fourth 20%", "Highest 20%"]
ses_select = alt.selection_point(fields=["SES_Quintile"], name="ses_select")

left_chart = alt.Chart(gpa_data).mark_bar(cornerRadiusTopLeft=4, cornerRadiusTopRight=4, cursor="pointer").encode(
    x=alt.X("SES_Quintile:N", title=None, sort=ses_order, axis=alt.Axis(labelAngle=-45)),
    y=alt.Y("GPA:Q", title="Average 12th Grade GPA", scale=alt.Scale(domain=[2.0, 3.5])),
    color=alt.value("#4c8dff"),
    opacity=alt.condition(ses_select, alt.value(1), alt.value(0.3)),
    tooltip=["SES_Quintile:N", alt.Tooltip("GPA:Q", format=".2f"), alt.Tooltip("Count:Q", format=",d")]
).add_params(ses_select).properties(
    name="view_1",
    title={"text": "Click SES Level", "color": "#FFFFFF", "fontSize": 14},
    width=300, height=350
)

right_chart = alt.Chart(stem_props).mark_rect().encode(
    x=alt.X("STEM_Major:N", title=None),
    y=alt.Y("SES_Quintile:O", title="SES Quintile", sort=ses_order),
    color=alt.Color("Proportion:Q", scale=alt.Scale(scheme="blues"), title="% of Students"),
    tooltip=["SES_Quintile:N", "STEM_Major:N", alt.Tooltip("Proportion:Q", format=".1f")]
).transform_filter(ses_select).properties(
    title={"text": "STEM Major Outcome", "color": "#FFFFFF", "fontSize": 14},
    width=200, height=350
)

viz6 = alt.hconcat(left_chart, right_chart).resolve_scale(color="independent")
save_chart(viz6, "hsls_gpa_ses_trajectory.json")
viz6

---
## COMBINED VISUALIZATIONS (Viz 7-9)
---

### Viz 7: Combined Efficacy by Language (PISA US vs HSLS)

**HIGHLIGHT selection (not filter)**

**Left (Heat Map):** PISA US - Language x Gender -> z(Efficacy)  
**Right (Heat Map):** HSLS - Language x Gender -> z(Efficacy)

In [ ]:
pisa_us = pisa_df[(pisa_df["CNT"] == "USA") &
                   (pisa_df["MATHEFF"].notna()) & 
                   (pisa_df["LANGN"].notna()) &
                   (pisa_df["ST004D01T"].isin([1, 2]))].copy()
pisa_us["Gender"] = pisa_us["ST004D01T"].map({1: "Female", 2: "Male"})
pisa_us["Language_Group"] = np.where(pisa_us["LANGN"] == 313, "English", "Other Language")
pisa_us["Z_Efficacy"] = (pisa_us["MATHEFF"] - pisa_us["MATHEFF"].mean()) / pisa_us["MATHEFF"].std()

hsls_lang = hsls_df[(hsls_df["X1MTHEFF"].notna()) & 
                     (hsls_df["X1DUALLANG"].isin([1, 2])) &
                     (hsls_df["X1SEX"].isin([1, 2]))].copy()
hsls_lang["Gender"] = hsls_lang["X1SEX"].map({1: "Male", 2: "Female"})
hsls_lang["Language_Group"] = hsls_lang["X1DUALLANG"].map({1: "English Only", 2: "Dual Language"})
hsls_lang["Z_Efficacy"] = (hsls_lang["X1MTHEFF"] - hsls_lang["X1MTHEFF"].mean()) / hsls_lang["X1MTHEFF"].std()

pisa_heat = pisa_us.groupby(["Language_Group", "Gender"]).agg(Mean_Efficacy=("Z_Efficacy", "mean")).reset_index()
pisa_heat["Source"] = "PISA US"

hsls_heat = hsls_lang.groupby(["Language_Group", "Gender"]).agg(Mean_Efficacy=("Z_Efficacy", "mean")).reset_index()
hsls_heat["Source"] = "HSLS"

lang_highlight = alt.selection_point(fields=["Language_Group"], name="lang_highlight")

left_chart = alt.Chart(pisa_heat).mark_rect(cursor="pointer").encode(
    x=alt.X("Gender:N", title=None),
    y=alt.Y("Language_Group:N", title="Home Language (PISA US)"),
    color=alt.Color("Mean_Efficacy:Q", scale=alt.Scale(scheme="redblue", domain=[-0.5, 0.5]), title="z(Efficacy)"),
    opacity=alt.condition(lang_highlight, alt.value(1), alt.value(0.3)),
    tooltip=["Language_Group:N", "Gender:N", alt.Tooltip("Mean_Efficacy:Q", format=".3f")]
).add_params(lang_highlight).properties(
    name="view_1",
    title={"text": "PISA US (2022)", "color": "#FFFFFF", "fontSize": 14},
    width=200, height=200
)

right_chart = alt.Chart(hsls_heat).mark_rect().encode(
    x=alt.X("Gender:N", title=None),
    y=alt.Y("Language_Group:N", title="Language Background (HSLS)"),
    color=alt.Color("Mean_Efficacy:Q", scale=alt.Scale(scheme="redblue", domain=[-0.5, 0.5]), title="z(Efficacy)"),
    opacity=alt.condition(lang_highlight, alt.value(1), alt.value(0.3)),
    tooltip=["Language_Group:N", "Gender:N", alt.Tooltip("Mean_Efficacy:Q", format=".3f")]
).properties(
    title={"text": "HSLS (2009)", "color": "#FFFFFF", "fontSize": 14},
    width=200, height=200
)

viz7 = alt.hconcat(left_chart, right_chart).resolve_scale(color="shared")
save_chart(viz7, "combined_efficacy_comparison.json")
viz7

### Viz 8: Combined Belonging vs SES (PISA vs HSLS)

**HIGHLIGHT selection (not filter)**

**Left (Scatter + LOESS):** PISA - SES bin vs z(Belonging)  
**Right (Scatter + LOESS):** HSLS - SES bin vs z(Belonging)

In [ ]:
pisa_belong = pisa_df[(pisa_df["ESCS"].notna()) & 
                       (pisa_df["BELONG"].notna()) &
                       (pisa_df["ST004D01T"].isin([1, 2]))].copy()
pisa_belong["Gender"] = pisa_belong["ST004D01T"].map({1: "Female", 2: "Male"})
pisa_belong["SES_Bin"] = pd.qcut(pisa_belong["ESCS"], 5, labels=[1, 2, 3, 4, 5]).astype(int)
pisa_belong["Z_Belong"] = (pisa_belong["BELONG"] - pisa_belong["BELONG"].mean()) / pisa_belong["BELONG"].std()

hsls_belong = hsls_df[(hsls_df["X1SESQ5"].notna()) & 
                       (hsls_df["X1SCHOOLBEL"].notna()) &
                       (hsls_df["X1SEX"].isin([1, 2]))].copy()
hsls_belong = hsls_belong[hsls_belong["X1SESQ5"] > 0]
hsls_belong["Gender"] = hsls_belong["X1SEX"].map({1: "Male", 2: "Female"})
hsls_belong["SES_Bin"] = hsls_belong["X1SESQ5"].astype(int)
hsls_belong["Z_Belong"] = (hsls_belong["X1SCHOOLBEL"] - hsls_belong["X1SCHOOLBEL"].mean()) / hsls_belong["X1SCHOOLBEL"].std()

pisa_sample = pisa_belong[["SES_Bin", "Z_Belong", "Gender"]].sample(n=3000, random_state=42)
pisa_sample["Source"] = "PISA"

hsls_sample = hsls_belong[["SES_Bin", "Z_Belong", "Gender"]].sample(n=3000, random_state=42)
hsls_sample["Source"] = "HSLS"

ses_highlight = alt.selection_point(fields=["SES_Bin"], name="ses_highlight")

def make_scatter_loess(data, title):
    scatter = alt.Chart(data).mark_circle(opacity=0.3, size=20).encode(
        x=alt.X("SES_Bin:O", title="SES Quintile"),
        y=alt.Y("Z_Belong:Q", title="z(Belonging)", scale=alt.Scale(domain=[-3, 3])),
        color=alt.Color("Gender:N", scale=alt.Scale(domain=["Female", "Male"], range=["#E91E63", "#1976D2"]),
                        legend=alt.Legend(title="Gender", orient="top")),
        opacity=alt.condition(ses_highlight, alt.value(0.5), alt.value(0.1)),
        tooltip=["SES_Bin:O", "Gender:N", alt.Tooltip("Z_Belong:Q", format=".2f")]
    )
    loess = alt.Chart(data).mark_line(strokeWidth=3).encode(
        x=alt.X("SES_Bin:O"),
        y=alt.Y("mean_belong:Q"),
        color=alt.Color("Gender:N", scale=alt.Scale(domain=["Female", "Male"], range=["#E91E63", "#1976D2"]))
    ).transform_aggregate(mean_belong="mean(Z_Belong)", groupby=["SES_Bin", "Gender"])
    return (scatter + loess).properties(title={"text": title, "color": "#FFFFFF", "fontSize": 14}, width=350, height=300)

left_chart = make_scatter_loess(pisa_sample, "PISA (International)").add_params(ses_highlight)
left_chart.name = "view_1"

right_chart = make_scatter_loess(hsls_sample, "HSLS (US)")

viz8 = alt.hconcat(left_chart, right_chart).resolve_scale(color="shared")
save_chart(viz8, "combined_ses_achievement.json")
viz8

### Viz 9: Parent Education Premium (Combined)

**Driver (Line Chart - 3 lines):** z-Math by parent education, colored by Source  
**Driven (Vertical Bar):** Premium vs <HS baseline for selected level

In [ ]:
hsls_edu_map = {1: "Less than HS", 2: "HS Diploma", 3: "Some College", 4: "Some College", 5: "Bachelor's", 6: "Graduate+", 7: "Graduate+"}
pisa_edu_map = {1: "Less than HS", 2: "Less than HS", 3: "HS Diploma", 4: "Some College", 5: "Some College", 6: "Bachelor's", 7: "Graduate+", 8: "Graduate+"}

hsls_edu = hsls_df[(hsls_df["X1PAR1EDU"].notna()) & (hsls_df["X1TXMTSCOR"].notna())].copy()
hsls_edu = hsls_edu[hsls_edu["X1PAR1EDU"] > 0]
hsls_edu["Parent_Education"] = hsls_edu["X1PAR1EDU"].map(hsls_edu_map)
hsls_edu["Math_Z"] = (hsls_edu["X1TXMTSCOR"] - hsls_edu["X1TXMTSCOR"].mean()) / hsls_edu["X1TXMTSCOR"].std()
hsls_edu["Source"] = "HSLS (US 2009)"

pisa_edu = pisa_df[(pisa_df["HISCED"].notna()) & (pisa_df["PV1MATH"].notna())].copy()
pisa_edu = pisa_edu[pisa_edu["HISCED"] > 0]
pisa_edu["Parent_Education"] = pisa_edu["HISCED"].map(pisa_edu_map)
pisa_edu["Math_Z"] = (pisa_edu["PV1MATH"] - pisa_edu["PV1MATH"].mean()) / pisa_edu["PV1MATH"].std()

pisa_us_edu = pisa_edu[pisa_edu["CNT"] == "USA"].copy()
pisa_us_edu["Source"] = "PISA US (2022)"
pisa_intl_edu = pisa_edu[pisa_edu["CNT"] != "USA"].copy()
pisa_intl_edu["Source"] = "PISA Intl (2022)"

combined = pd.concat([hsls_edu[["Source", "Parent_Education", "Math_Z"]], 
                      pisa_us_edu[["Source", "Parent_Education", "Math_Z"]], 
                      pisa_intl_edu[["Source", "Parent_Education", "Math_Z"]]])

line_data = combined.groupby(["Parent_Education", "Source"]).agg(Math_Score=("Math_Z", "mean")).reset_index()

baseline = line_data[line_data["Parent_Education"] == "Less than HS"].set_index("Source")["Math_Score"].to_dict()
line_data["Premium"] = line_data.apply(lambda r: r["Math_Score"] - baseline.get(r["Source"], 0), axis=1)

edu_order = ["Less than HS", "HS Diploma", "Some College", "Bachelor's", "Graduate+"]
source_order = ["HSLS (US 2009)", "PISA US (2022)", "PISA Intl (2022)"]
source_colors = ["#E69F00", "#56B4E9", "#009E73"]

edu_select = alt.selection_point(fields=["Parent_Education"], name="edu_select")

left_chart = alt.Chart(line_data).mark_line(point={"filled": True, "size": 80}, strokeWidth=3, cursor="pointer").encode(
    x=alt.X("Parent_Education:O", title="Parent Education", sort=edu_order, axis=alt.Axis(labelAngle=-45)),
    y=alt.Y("Math_Score:Q", title="Standardized Math Score", scale=alt.Scale(domain=[-1.0, 1.0])),
    color=alt.Color("Source:N", scale=alt.Scale(domain=source_order, range=source_colors),
                    legend=alt.Legend(title="Data Source", orient="top")),
    opacity=alt.condition(edu_select, alt.value(1), alt.value(0.3)),
    tooltip=["Parent_Education:N", "Source:N", alt.Tooltip("Math_Score:Q", format=".3f")]
).add_params(edu_select).properties(
    name="view_1",
    title={"text": "Click Education Level", "color": "#FFFFFF", "fontSize": 14},
    width=450, height=350
)

right_chart = alt.Chart(line_data).mark_bar(cornerRadiusTopLeft=4, cornerRadiusTopRight=4).encode(
    x=alt.X("Source:N", title=None, sort=source_order),
    y=alt.Y("Premium:Q", title="Premium vs <HS Baseline", scale=alt.Scale(domain=[-0.2, 1.2])),
    color=alt.Color("Source:N", scale=alt.Scale(domain=source_order, range=source_colors),
                    legend=alt.Legend(title="Data Source", orient="top")),
    tooltip=["Parent_Education:N", "Source:N", alt.Tooltip("Premium:Q", format=".3f")]
).transform_filter(edu_select).properties(
    title={"text": "Education Premium", "color": "#FFFFFF", "fontSize": 14},
    width=250, height=350
)

viz9 = alt.hconcat(left_chart, right_chart).resolve_scale(color="independent")
save_chart(viz9, "combined_parent_education.json")
viz9

---
## Summary

All 9 visualizations redesigned with restricted chart types:

| Viz | Driver | Driven | Interaction |
|-----|--------|--------|-------------|
| 1 | Horizontal Bar | Line | Filter |
| 2 | Heat Map | Scatter + LOESS | Filter |
| 3 | Vertical Bar | Scatter | Filter |
| 4 | Line (2 lines) | Vertical Bar | Filter |
| 5 | Horizontal Bar | Scatter + regression | Filter |
| 6 | Vertical Bar | Heat Map | Filter |
| 7 | Heat Map | Heat Map | Highlight |
| 8 | Scatter + LOESS | Scatter + LOESS | Highlight |
| 9 | Line (3 lines) | Vertical Bar | Filter |

In [ ]:
import os
json_files = [
    "pisa_gender_efficacy_dumbbell.json",
    "pisa_anxiety_performance_heatmap.json",
    "combined_immigration.json",
    "combined_gender_stem.json",
    "hsls_math_identity_race.json",
    "hsls_gpa_ses_trajectory.json",
    "combined_efficacy_comparison.json",
    "combined_ses_achievement.json",
    "combined_parent_education.json"
]

print("Generated JSON files:")
for f in json_files:
    path = OUTPUT_DIR / f
    if path.exists():
        size = os.path.getsize(path)
        print(f"  {f}: {size/1024:.1f} KB")
    else:
        print(f"  {f}: NOT FOUND")